# ELW 실시간 호가 수집기
키움 Open API를 사용하여 ELW 실시간 호가 및 그리스 지표를 CSV로 저장

## 1. 설정

In [1]:
# === 사용자 설정 ===
STOCK_CODE = "52L971"  # 종목코드 (예: 52L971 - 미래L791삼성전자콜)
SCREEN_NO = "1000"     # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop

## 3. CSV 저장 설정

In [ ]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"{STOCK_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의
CSV_COLUMNS = [
    "timestamp", "time", "code", "current_price",
    "ask_price_1", "ask_qty_1", "ask_price_2", "ask_qty_2", 
    "ask_price_3", "ask_qty_3", "ask_price_4", "ask_qty_4", 
    "ask_price_5", "ask_qty_5",
    "bid_price_1", "bid_qty_1", "bid_price_2", "bid_qty_2", 
    "bid_price_3", "bid_qty_3", "bid_price_4", "bid_qty_4", 
    "bid_price_5", "bid_qty_5",
    "volume",
    "theoretical_price", "iv", "delta", "gamma", "theta", "vega", "rho", "lp_iv"
]

# FID 매핑 (호가, 체결, 그리스 지표)
FID_MAP = {
    # 체결 데이터
    "time": 20, "current_price": 10, "volume": 15,
    # 매도호가/잔량
    "ask_price_1": 41, "ask_price_2": 42, "ask_price_3": 43, "ask_price_4": 44, "ask_price_5": 45,
    "ask_qty_1": 61, "ask_qty_2": 62, "ask_qty_3": 63, "ask_qty_4": 64, "ask_qty_5": 65,
    # 매수호가/잔량
    "bid_price_1": 51, "bid_price_2": 52, "bid_price_3": 53, "bid_price_4": 54, "bid_price_5": 55,
    "bid_qty_1": 71, "bid_qty_2": 72, "bid_qty_3": 73, "bid_qty_4": 74, "bid_qty_5": 75,
    # 그리스 지표
    "theoretical_price": 670, "iv": 671, "delta": 672, "gamma": 673,
    "theta": 674, "vega": 675, "rho": 676, "lp_iv": 706
}

# 실시간 타입별 FID 그룹
FID_GROUPS = {
    "quote": ["ask_price_1", "ask_price_2", "ask_price_3", "ask_price_4", "ask_price_5",
              "ask_qty_1", "ask_qty_2", "ask_qty_3", "ask_qty_4", "ask_qty_5",
              "bid_price_1", "bid_price_2", "bid_price_3", "bid_price_4", "bid_price_5",
              "bid_qty_1", "bid_qty_2", "bid_qty_3", "bid_qty_4", "bid_qty_5"],
    "trade": ["time", "current_price", "volume"],
    "greeks": ["theoretical_price", "iv", "delta", "gamma", "theta", "vega", "rho", "lp_iv"]
}

## 4. KiwoomAPI 클래스 정의

In [ ]:
class KiwoomELWCollector:
    def __init__(self, stock_code, screen_no, csv_path):
        self.stock_code = stock_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        
        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")
        
        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)
        
        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0
        
        # 마지막 수신 데이터 캐시 (각 타입별로 수신된 값을 유지)
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["code"] = stock_code
        
        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()
        
        # CSV 헤더 작성
        self._init_csv()
    
    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")
    
    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()
    
    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True
            
            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")
            
            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False
        
        self.login_event_loop.exit()
    
    def set_real_reg(self):
        """실시간 등록"""
        # 등록할 FID 목록 (모든 FID 포함)
        fid_list = ";".join([str(fid) for fid in FID_MAP.values()])
        
        result = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.stock_code, fid_list, "0"
        )
        return result
    
    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value
    
    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 및 CSV 저장"""
        
        # [디버깅] 새로운 real_type이 들어오면 출력
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' <<<\n")
        
        # 처리할 실시간 타입 목록 (띄어쓰기 있는 버전 + 없는 버전 모두 포함)
        valid_types = (
            "주식호가잔량", "ELW호가잔량",      # 호가 데이터
            "주식체결", "ELW체결",              # 체결 데이터 (현재가, 거래량)
            "ELW이론가", "ELW 이론가",          # 이론가, IV (띄어쓰기 양쪽 버전)
            "ELW지표", "ELW 지표",              # Greeks (띄어쓰기 양쪽 버전)
        )
        
        if real_type not in valid_types:
            return
        
        # 타입별로 해당하는 FID만 조회하여 캐시 업데이트
        if real_type in ("주식호가잔량", "ELW호가잔량"):
            # 호가/잔량 데이터
            fid_keys = [
                "ask_price_1", "ask_price_2", "ask_price_3", "ask_price_4", "ask_price_5",
                "ask_qty_1", "ask_qty_2", "ask_qty_3", "ask_qty_4", "ask_qty_5",
                "bid_price_1", "bid_price_2", "bid_price_3", "bid_price_4", "bid_price_5",
                "bid_qty_1", "bid_qty_2", "bid_qty_3", "bid_qty_4", "bid_qty_5",
            ]
        elif real_type in ("주식체결", "ELW체결"):
            # 체결 데이터
            fid_keys = ["time", "current_price", "volume"]
        elif real_type in ("ELW이론가", "ELW 이론가"):
            # 이론가, IV
            fid_keys = ["theoretical_price", "iv"]
        elif real_type in ("ELW지표", "ELW 지표"):
            # Greeks
            fid_keys = ["delta", "gamma", "theta", "vega", "rho", "lp_iv"]
        else:
            fid_keys = []
        
        # 해당 타입의 FID 값 조회 및 캐시 업데이트
        for key in fid_keys:
            if key in FID_MAP:
                value = self.get_real_data(code, FID_MAP[key])
                # 부호 제거 (가격 데이터의 +/- 기호)
                if value and (value.startswith("+") or value.startswith("-")):
                    value = value[1:]
                if value:  # 빈 값이 아닌 경우에만 업데이트
                    self.last_data[key] = value
        
        # 타임스탬프 업데이트
        self.last_data["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        self.last_data["code"] = code
        
        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")
        
        # 콘솔 출력
        print(f"[{self.data_count}] {real_type:12} | "
              f"현재가: {self.last_data.get('current_price', ''):>8} | "
              f"거래량: {self.last_data.get('volume', ''):>8} | "
              f"델타: {self.last_data.get('delta', ''):>8} | "
              f"IV: {self.last_data.get('iv', ''):>8}")
    
    def run(self):
        """메인 실행"""
        self.login()
        
        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return
        
        print("\n" + "=" * 50)
        print(f"ELW 실시간 호가 등록 중... (종목: {self.stock_code})")
        print("=" * 50)
        
        result = self.set_real_reg()
        print(f"실시간 등록 결과: {result}")
        
        print("\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print("\n수신 타입: 주식체결, ELW체결, ELW호가잔량, ELW 이론가, ELW 지표")
        print("[디버깅] 새로운 real_type 수신 시 자동 출력됩니다.\n")
        
        self.app.exec_()
    
    def stop(self):
        """수집 중지"""
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomELWCollector(
    stock_code=STOCK_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH
)

# 실행 (중지하려면 Kernel > Interrupt 또는 Ctrl+C)
collector.run()

CSV 파일 생성: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\project1\data\52L971_20260311_1302.csv

[성공] 로그인 완료
사용자: 박상혁
서버: 실투자

ELW 실시간 호가 등록 중... (종목: 52L971)
실시간 등록 결과: 0

실시간 데이터 수집 중... (중지: Kernel Interrupt)
저장 위치: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\project1\data\52L971_20260311_1302.csv

수신 타입: 주식체결, ELW체결, ELW호가잔량, ELW이론가

[1] 주식호가잔량       | 현재가:          | 거래량:          | 델타:          | IV:         
[2] 주식호가잔량       | 현재가:          | 거래량:          | 델타:          | IV:         
[3] 주식호가잔량       | 현재가:          | 거래량:          | 델타:          | IV:         
[4] 주식호가잔량       | 현재가:          | 거래량:          | 델타:          | IV:         
[5] 주식호가잔량       | 현재가:          | 거래량:          | 델타:          | IV:         


## 6. 수집된 데이터 확인

In [ ]:
# 수집된 데이터 로드 및 미리보기
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print(f"총 {len(df)}개 데이터")
df.head(10)

In [ ]:
# 데이터 타입 확인
df.info()